# Import

In [1]:
import math

In [2]:
# --- Helper Functions (เครื่องมือช่วยแทน Numpy) ---
def zeros(n):
    return [0.0] * n

def identity_matrix(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

def deep_copy_matrix(A):
    # Copy แบบแยกข้อมูลขาดจากกัน (Deep Copy)
    return [row[:] for row in A]

def print_matrix(A):
    for row in A:
        print(row)



# Input Functions

In [3]:
# --- Input Functions ---
def input_matrix():
    while True:
        try:
            n_input = input("\nกรอกขนาดของ Matrix A (n x n): ")
            if not n_input: return None # Handle cancel
            n = int(n_input)
            if n <= 0:
                print("⚠️ n ต้องมากกว่า 0")
                continue
            
            A = []
            print("กรอกค่า Matrix A ทีละแถว (เว้นวรรคระหว่างตัวเลข):")
            for i in range(n):
                while True:
                    try:
                        row_str = input(f"แถวที่ {i+1}: ").split()
                        if len(row_str) != n:
                            print(f"⚠️ ต้องกรอก {n} ตัว")
                            continue
                        row = [float(x) for x in row_str]
                        A.append(row)
                        break
                    except ValueError:
                         print("⚠️ กรุณากรอกตัวเลขเท่านั้น")
            return A
        except ValueError:
            print("⚠️ กรุณากรอกจำนวนเต็ม")

def input_vector(n):
    while True:
        try:
            b_str = input("กรอก vector b: ").split()
            if len(b_str) != n:
                print(f"⚠️ vector b ต้องมี {n} ตัว")
            else:
                return [float(x) for x in b_str]
        except ValueError:
            print("⚠️ กรุณากรอกตัวเลขเท่านั้น")

# Rank via Elimination หาลำดับขั้นในการ elimination

In [4]:
def rank_matrix(M):
    # ทำ Deep copy เพื่อไม่ให้กระทบ Matrix ต้นฉบับ
    A = deep_copy_matrix(M)
    rows = len(A)
    cols = len(A[0])
    rank = 0

    for col in range(cols):
        if rank >= rows: break
        
        # หา Pivot ที่ไม่ใช่ 0
        pivot_row = -1
        for r in range(rank, rows):
            if abs(A[r][col]) > 1e-10:
                pivot_row = r
                break
        
        if pivot_row != -1:
            # สลับแถว (Swap rows)
            A[rank], A[pivot_row] = A[pivot_row], A[rank]
            
            # Normalize แถว Pivot ให้เป็น 1
            pivot_val = A[rank][col]
            A[rank] = [x / pivot_val for x in A[rank]]
            
            # Eliminate แถวอื่นๆ
            for r in range(rows):
                if r != rank:
                    factor = A[r][col]
                    # Row Operation: R_r = R_r - factor * R_rank
                    A[r] = [curr - factor * piv for curr, piv in zip(A[r], A[rank])]
            
            rank += 1
            
    return rank

# Consistency Check

In [5]:
def check_solution_type(A, b):
    # สร้าง Augmented Matrix [A|b] แบบ manual
    Ab = [row + [val] for row, val in zip(A, b)]
    
    rankA = rank_matrix(A)
    rankAb = rank_matrix(Ab)
    n = len(A[0]) # จำนวนตัวแปร

    if rankA < rankAb:
        return "No" # No Solution
    elif rankA < n:
        return "Infinite" # Infinite Solutions
    else:
        return "Unique" # Unique Solution

# Solution Reporter

In [6]:
def print_result(method_name, x, status):
    print(f"\n[{method_name}]")
    if status == "Unique" and x is not None:
        print("✅ ระบบสมการนี้มีคำตอบเดียว")
        print("x =", [round(val, 4) for val in x]) # ปัดเศษแสดงผลสวยๆ
    elif status == "Infinite":
        print("⚠️ ระบบสมการนี้มีคำตอบไม่สิ้นสุด (Infinite Solutions)")
    elif status == "No":
        print("❌ ระบบสมการนี้ไม่มีคำตอบ (No Solution)")
    else:
        print("❌ ไม่สามารถหาคำตอบได้")

# Gaussian Elimination

In [7]:
def gauss_elimination_pivot(A_in, b):
    n = len(b)
    # สร้าง Augmented Matrix [A|b]
    Ab = [row[:] + [b[i]] for i, row in enumerate(A_in)]
    
    # Forward Elimination
    for i in range(n):
        # Partial Pivoting
        max_row = i
        for r in range(i+1, n):
            if abs(Ab[r][i]) > abs(Ab[max_row][i]):
                max_row = r
        Ab[i], Ab[max_row] = Ab[max_row], Ab[i]
        
        if abs(Ab[i][i]) < 1e-10: return None # Singular
        
        for r in range(i+1, n):
            factor = Ab[r][i] / Ab[i][i]
            # Row operation: Ab[r] = Ab[r] - factor * Ab[i]
            Ab[r] = [x - factor * y for x, y in zip(Ab[r], Ab[i])]
            
    # Back Substitution
    x = zeros(n)
    for i in range(n-1, -1, -1):
        # sum_ax = sum(A[i][j] * x[j] for j > i)
        sum_ax = sum(Ab[i][j] * x[j] for j in range(i+1, n))
        x[i] = (Ab[i][-1] - sum_ax) / Ab[i][i]
        
    return x

# Gauss Jordan

In [8]:
def gauss_jordan(A_in, b):
    n = len(b)
    Ab = [row[:] + [b[i]] for i, row in enumerate(A_in)]
    
    for i in range(n):
        if abs(Ab[i][i]) < 1e-10: return None
        
        # ทำให้ Pivot เป็น 1
        pivot = Ab[i][i]
        Ab[i] = [x / pivot for x in Ab[i]]
        
        # ทำให้แถวอื่นในคอลัมน์นี้เป็น 0
        for r in range(n):
            if r != i:
                factor = Ab[r][i]
                Ab[r] = [curr - factor * piv for curr, piv in zip(Ab[r], Ab[i])]
                
    # คำตอบคือคอลัมน์สุดท้าย
    return [row[-1] for row in Ab]

# LU Factorization

In [9]:
def lu_factorization(A_in, b):
    n = len(A_in)
    L = identity_matrix(n)
    U = deep_copy_matrix(A_in) # U เริ่มต้นคือ A แล้วค่อยๆ เปลี่ยน
    
    # Decompose A -> L, U
    for i in range(n):
        if abs(U[i][i]) < 1e-10: return None
        
        for r in range(i+1, n):
            factor = U[r][i] / U[i][i]
            L[r][i] = factor
            # Update row r of U
            U[r] = [curr - factor * piv for curr, piv in zip(U[r], U[i])]
            
    # Solve Ly = b (Forward Substitution)
    y = zeros(n)
    for i in range(n):
        sum_ly = sum(L[i][j] * y[j] for j in range(i))
        y[i] = b[i] - sum_ly
        
    # Solve Ux = y (Backward Substitution)
    x = zeros(n)
    for i in range(n-1, -1, -1):
        sum_ux = sum(U[i][j] * x[j] for j in range(i+1, n))
        x[i] = (y[i] - sum_ux) / U[i][i]
        
    return x

# Multiply Vector

In [10]:
def multiply_matrix_vector(A, b):
    n = len(b)
    result = []
    for i in range(n):
        # Dot product ของแถว A[i] กับ vector b
        val = sum(A[i][j] * b[j] for j in range(n))
        result.append(val)
    return result

# Inverse Matrix

In [11]:
def multiply_matrix_vector(A, b):
    n = len(b)
    result = []
    for i in range(n):
        # Dot product ของแถว A[i] กับ vector b
        val = sum(A[i][j] * b[j] for j in range(n))
        result.append(val)
    return result

def inverse_matrix(A_in):
    n = len(A_in)
    # สร้าง [A | I]
    I = identity_matrix(n)
    AI = [row[:] + I[i] for i, row in enumerate(A_in)]
    
    # Gauss-Jordan
    for i in range(n):
        if abs(AI[i][i]) < 1e-10: return None
        
        pivot = AI[i][i]
        AI[i] = [x / pivot for x in AI[i]]
        
        for r in range(n):
            if r != i:
                factor = AI[r][i]
                AI[r] = [curr - factor * piv for curr, piv in zip(AI[r], AI[i])]
    
    # ตัดเอาเฉพาะครึ่งหลัง (Inverse Matrix)
    invA = []
    for row in AI:
        invA.append(row[n:])
    return invA

# Helper สำหรับหา Determinant (เพื่อเช็ค Singular แบบเร็วๆ ใน Main)
def get_determinant(A_in):
    # ใช้ Gaussian เพื่อแปลงเป็น Upper Triangular แล้วคูณเส้นทแยงมุม
    A = deep_copy_matrix(A_in)
    n = len(A)
    det = 1.0
    for i in range(n):
        pivot = i
        while pivot < n and abs(A[pivot][i]) < 1e-10:
            pivot += 1
        if pivot == n: return 0.0 # Pivot เป็น 0 ทั้งคอลัมน์
        if pivot != i:
            A[i], A[pivot] = A[pivot], A[i]
            det *= -1 # สลับแถว det กลับเครื่องหมาย
            
        det *= A[i][i]
        for r in range(i+1, n):
            factor = A[r][i] / A[i][i]
            A[r] = [curr - factor * piv for curr, piv in zip(A[r], A[i])]
    return det

# INPUT FUNCTIONS

In [12]:
while True:
    print("\n==============================")
    print("โปรแกรมแก้ระบบสมการเชิงเส้น")
    print("1. กรอกสมการใหม่")
    print("0. จบการทำงาน")
    print("==============================")
    
    choice = input("เลือกเมนู: ")
    
    if choice == '0':
        print("จบการทำงานของโปรแกรม")
        break
        
    elif choice == '1':
        A = input_matrix()
        if A is None: continue
        n = len(A)
        
        # เช็ค Singular แบบเร็วๆ โดยใช้ Determinant ที่เขียนเอง
        det = get_determinant(A)
        if abs(det) < 1e-12:
            print(f"\n⚠️ Matrix นี้อาจเป็น Singular (det ≈ {det:.4f})")
            print("บาง method อาจไม่สามารถคำนวณได้")
            
        has_b = input("Matrix นี้มี vector b หรือไม่ (y/n): ").lower()
        
        if has_b == 'y':
            b = input_vector(n)
            status = check_solution_type(A, b)
            
            # 1. Gauss Elimination
            x_gauss = gauss_elimination_pivot(A, b)
            print_result("Gauss Elimination with Pivoting", x_gauss, status)
            
            # 2. Gauss-Jordan
            x_jordan = gauss_jordan(A, b)
            print_result("Gauss Jordan Elimination", x_jordan, status)
            
            # 3. LU
            x_lu = lu_factorization(A, b)
            print_result("LU Factorization", x_lu, status)
            
            # 4. Inverse Check
            print("\n[Inverse Matrix และ x จาก A⁻¹b]")
            invA = inverse_matrix(A)
            
            if invA is None:
                print("❌ ไม่สามารถหา Inverse Matrix ได้ (Matrix เป็น Singular)")
            else:
                print("✅ Inverse Matrix =")
                for row in invA:
                    print([round(val, 4) for val in row])
                
                x_inv = multiply_matrix_vector(invA, b)
                print("✅ x = A⁻¹b =", [round(val, 4) for val in x_inv])
                
        elif has_b == 'n':
            print("\n[Inverse Matrix]")
            invA = inverse_matrix(A)
            if invA is None:
                print("❌ ไม่สามารถหา Inverse Matrix ได้ (Matrix เป็น Singular)")
            else:
                print("✅ Inverse Matrix =")
                for row in invA:
                    print([round(val, 4) for val in row])
        else:
            print("⚠️ กรุณาเลือก y หรือ n เท่านั้น")
            
    else:
        print("⚠️ เลือกเมนูไม่ถูกต้อง กรุณาเลือกใหม่")


โปรแกรมแก้ระบบสมการเชิงเส้น
1. กรอกสมการใหม่
0. จบการทำงาน
กรอกค่า Matrix A ทีละแถว (เว้นวรรคระหว่างตัวเลข):
⚠️ กรุณาเลือก y หรือ n เท่านั้น

โปรแกรมแก้ระบบสมการเชิงเส้น
1. กรอกสมการใหม่
0. จบการทำงาน
กรอกค่า Matrix A ทีละแถว (เว้นวรรคระหว่างตัวเลข):

[Gauss Elimination with Pivoting]
✅ ระบบสมการนี้มีคำตอบเดียว
x = [9.8732, -2.507, 4.7746, 1.0704]

[Gauss Jordan Elimination]
✅ ระบบสมการนี้มีคำตอบเดียว
x = [9.8732, -2.507, 4.7746, 1.0704]

[LU Factorization]
✅ ระบบสมการนี้มีคำตอบเดียว
x = [9.8732, -2.507, 4.7746, 1.0704]

[Inverse Matrix และ x จาก A⁻¹b]
✅ Inverse Matrix =
[0.493, 0.7465, -0.2676, -0.0704]
[-0.0282, -0.0141, -0.0704, -0.2817]
[0.0986, 0.5493, -0.2535, -0.0141]
[0.2817, 0.1408, -0.2958, -0.1831]
✅ x = A⁻¹b = [9.8732, -2.507, 4.7746, 1.0704]

โปรแกรมแก้ระบบสมการเชิงเส้น
1. กรอกสมการใหม่
0. จบการทำงาน
จบการทำงานของโปรแกรม
